<a href="https://colab.research.google.com/github/mirian2004/AI-AI-/blob/Day18/Day18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DB 연동 API 호출**

Gemini API에 가상 DB 데이터를 만드는 실습

**가상 DB 생성**

먼저 코드로 스키마와 테이블을 정의해 가상 DB를 생성함

In [2]:
import sqlite3
import pandas as pd

# [1] 데이터베이스 파일 생성 및 연결
# 'shopping_mall.db'라는 파일 생성
conn = sqlite3.connect('shopping_mall.db')
cursor = conn.cursor()

# 초기화를 위해 기존 테이블이 있다면 삭제
cursor.execute("DROP TABLE IF EXISTS users")
cursor.execute("DROP TABLE IF EXISTS orders")

# [2] 테이블 만들기 (스키마 정의)
# 사용자 테이블 (users) : 고객 정보
cursor.execute('''
CREATE TABLE users(
  user_id TEXT PRIMARY KEY,
  name TEXT,
  style_preference TEXT
)
''')

cursor.execute('''
CREATE TABLE orders (
  order_id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id TEXT,
  product_name TEXT,
  purchase_date TEXT
)
''')

# [3] 더미 데이터 넣기 (샘플 데이터 적재)
# 사용자 데이터
users_data = [
    ('user_01', '김철수', '활동적이고 스포티한 스타일'),
    ('user_02', '이영희', '차분하고 모던한 오피스룩'),
    ('user_03', '박민수', '화려하고 힙한 스트릿 패션')
]
cursor.executemany("INSERT INTO users VALUES (?, ?, ?)", users_data)

# 주문 데이터
orders_data = [
    ('user_01', '나이키 러닝화', '2025-10-15'),
    ('user_01', '스포츠 땀방지 밴드', '2025-11-02'),
    ('user_02', '베이지색 슬랙스', '2025-10-20'),
    ('user_03', '오버핏 후드티', '2025-11-05')
]
cursor.executemany("INSERT INTO orders (user_id, product_name, purchase_date) VALUES (?, ?, ?)", orders_data)

# 변경사항 저장
conn.commit()

print("가상 데이터베이스 구축 완료! (shopping_mall.db)")

가상 데이터베이스 구축 완료! (shopping_mall.db)


**서버 연결 설정**

API Key를 설정해 Gemini API 서버와 연결

In [3]:
import os
from google import genai
from google.genai import types
from google.colab import userdata

# API 키 설정
os.environ["GEMINI_API_KEY"] = userdata.get('gemini_api_key')
client = genai.Client()

print("Google Gemini AI 서버 연결 설정 완료!")

Google Gemini AI 서버 연결 설정 완료!


**백엔드 전처리(DB 조회)**

DB에 개발자가 미리 지정해놓은 SQL을 실행해 '우리' 데이터를 조회함
-get_personalized_recommendation 함수에서 백엔드 로직을 수행

**백엔드 전처리 (데이터 주입)**

DB에서 조회한 고객의 개인정보를 프롬프트에 주입해 개인화된 응답을 AI에게 요청

**API 요청 전송**

완성된 프롬프트와 함꼐 파라미터를 설정하고 API 요청을 전송

In [4]:
def get_personalized_recommendation(target_user_id):
    """
    1. DB에서 사용자 정보와 주문 내역을 조회 (SQL)
    2. 조회된 데이터를 프롬프트에 주입 (Data Injection)
    3. AI에게 추천 요청 (Gemini API Call)
    """

    # 1. [SQL 실행] DB에서 데이터 조회
    # 사용자 정보 조회
    cursor.execute("SELECT name, style_preference FROM users WHERE user_id = ?", (target_user_id,))
    user_info = cursor.fetchone()

    if not user_info:
        return "존재하지 않는 고객 ID입니다."

    user_name = user_info[0]
    user_style = user_info[1]

    # 구매 내역 조회
    cursor.execute("SELECT product_name FROM orders WHERE user_id = ?", (target_user_id,))
    order_rows = cursor.fetchall()

    # 구매한 물건들을 리스트에서 텍스트로 변환
    purchased_items = ", ".join([row[0] for row in order_rows])
    if not purchased_items:
        purchased_items = "구매 이력 없음"

    print(f"[DB 조회 결과] 고객명: {user_name} / 스타일: {user_style} / 구매품: {purchased_items}")

    # 2. 데이터 주입 (DB에서 조회한 결과 주입)
    # 시스템 프롬프트 (페르소나)
    system_prompt = "너는 전문 패션 스타일리스트 AI야. 고객의 데이터를 분석해서 최고의 상품 하나를 추천해줘."

    # 사용자 프롬프트 작성 (DB 데이터 주입)
    user_prompt_template = f"""
    아래는 우리 고객의 데이터야. 분석 후 다음에 구매할 만한 상품 1개를 추천해주고 이유를 설명해줘.

    ---
    [고객 프로필]
    - 이름: {user_name}
    - 선호 스타일: {user_style}

    [최근 구매 내역]
    - {purchased_items}
    ---

    추천 상품:
    추천 이유:
    """

    # 3. [API 호출] Gemini API로 호출
    try:
        response = client.models.generate_content(
            model='gemini-flash-latest',
            contents=user_prompt_template,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt, # 시스템 프롬프트 설정
                temperature=0.7, # 온도 조절
                max_output_tokens=5000 # 토큰 길이 조
            )
        )
        return response.text

    except Exception as e:
        return f"AI 호출 중 오류 발생: {str(e)}"

**테스트**

고객의 시나리오를 예상해 테스트 진행. 필요에 따라 파라미터를 조정해 최적의 응답을 생성함

In [5]:
# [시나리오 1] 운동을 좋아하는 김철수 고객 (user_01)
print("--------------- [ CASE 1 ] ---------------")
result_1 = get_personalized_recommendation('user_01')
print("\nAI의 추천 결과:")
print(result_1)

print("\n" + "="*50 + "\n")

# [시나리오 2] 오피스룩을 입는 이영희 고객 (user_02)
print("--------------- [ CASE 2 ] ---------------")
result_2 = get_personalized_recommendation('user_02')
print("\nAI의 추천 결과:")
print(result_2)

# [시나리오 3] 존재하지 않는 고객
print("\n" + "="*50 + "\n")
print("--------------- [ CASE 3 ] ---------------")
print(get_personalized_recommendation('user_99'))

# 실습 종료 후 DB 연결 닫기
conn.close()

--------------- [ CASE 1 ] ---------------
[DB 조회 결과] 고객명: 김철수 / 스타일: 활동적이고 스포티한 스타일 / 구매품: 나이키 러닝화, 스포츠 땀방지 밴드

AI의 추천 결과:
**추천 상품:** 나이키 드라이핏 경량 러닝 바람막이 재킷

**추천 이유:** 
김철수 고객님은 활동적이고 스포티한 스타일을 선호하며, 최근 러닝화와 땀방지 밴드를 구매하셨습니다. 이는 본격적인 러닝이나 야외 운동을 즐기고 계심을 나타냅니다. 

이 재킷을 추천하는 이유는 다음과 같습니다.
1. **착장의 완성을 위한 필수 아이템:** 이미 갖추신 러닝화 및 땀방지 밴드와 함께 매치하면 완벽한 스포티 러닝 룩(Full-Set)을 연출할 수 있습니다.
2. **기능성과 스타일의 조화:** 운동 중 체온 조절과 땀 배출이 뛰어난 기능성 소재로, 고객님의 활동적인 라이프스타일에 최적화되어 있습니다.
3. **높은 활용도:** 야외 운동 시에는 물론, 일상에서 가볍게 걸치는 '애슬레저 룩'으로도 스타일리시하게 활용할 수 있어 만족도가 매우 높을 것으로 예상됩니다.


--------------- [ CASE 2 ] ---------------
[DB 조회 결과] 고객명: 이영희 / 스타일: 차분하고 모던한 오피스룩 / 구매품: 베이지색 슬랙스

AI의 추천 결과:
**추천 상품:** 네이비 싱글 테일러드 자켓

**추천 이유:**
1. **완벽한 컬러 매칭 (Navy & Beige):** 최근 구매하신 '베이지색 슬랙스'와 가장 클래식하면서도 세련되게 어울리는 컬러는 단연 '네이비'입니다. 네이비와 베이지의 조합은 차분함과 지적인 이미지를 동시에 전달하여 오피스룩의 정석으로 불립니다.
2. **고객 선호 스타일 반영:** 깔끔하게 떨어지는 테일러드 라인의 자켓은 이영희 님이 선호하시는 '차분하고 모던한 오피스룩'의 완성도를 높여주는 필수 아이템입니다.
3. **높은 활용도:** 베이지 슬랙스 외에도 화이트 셔츠나 기본 블라우스 위에 툭 걸치는 것